# Phase 1: Load, Explore & Clean
## **Student:** Eyas Alghamdi
## **Date:** April 2026

---

### **Project Overview**
This notebook follows the **'Day 8: Data Cleaning'** principles from our course materials. 

* **AI Assistance:** I used **ChatGPT** to refine the outlier detection logic and **Gemini** to verify the standard treatment for missing values in the Titanic dataset.
* **Goal:** Ingest the raw data, handle nulls, and prepare a clean baseline for feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Titanic Dataset
# Using the raw URL ensures the code runs directly on GitHub/Colab without local files
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
df = pd.read_csv(url)

print("Dataset Loaded Successfully!")
print(f"Shape: {df.shape}")
df.head()

### **Data Cleaning Steps**

Following **Day 8** guidelines, we must fix data types and handle missing values.
1.  **Fix Types:** Converting `pclass` to a string as it is a category.
2.  **Missing Values:** Filling `age` with the median and `embarked` with the mode.
3.  **Outliers:** Capping `fare` at the 99th percentile.

In [ ]:
# Fix Data Types
df['pclass'] = df['pclass'].astype(str)

# Handle Missing Values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Drop 'deck' as it has too many missing values (Ref: Day 8 Activity)
if 'deck' in df.columns:
    df.drop(columns=['deck'], inplace=True)

# Remove Duplicates
df.drop_duplicates(inplace=True)

print("Cleaning Complete. Missing values remaining:")
print(df.isnull().sum().sum())

### **Outlier Detection & Capping**
We use the **99th percentile cap** for the 'fare' column to prevent extreme luxury ticket prices from warping our future analysis.

In [ ]:
upper_limit = df['fare'].quantile(0.99)
df['fare'] = np.where(df['fare'] > upper_limit, upper_limit, df['fare'])

print(f"Fare capped at: {upper_limit}")

### **Modular Cleaning Function**
As required by the **Week 1 Modular Design** assessment, here is a reusable function for the cleaning process.

In [ ]:
def clean_data(raw_df):
    data = raw_df.copy()
    data['age'] = data['age'].fillna(data['age'].median())
    limit = data['fare'].quantile(0.99)
    data['fare'] = np.where(data['fare'] > limit, limit, data['fare'])
    return data

# Final Validation Checks
assert df['age'].isnull().sum() == 0, "Validation Error: Age still has nulls"
print("Final Validation Passed!")

# Save clean data for Phase 2
df.to_csv('titanic_cleaned.csv', index=False)